# Day-2 pilots: conformal prediction after compression (A) and under signal degradation (B)

**Purpose:** decide at the Nov 14 gate which topic becomes the paper. Nothing here is a paper result.

- **Pilot A:** does reusing the full-precision conformal threshold on a compressed model lose coverage? (PathMNIST)
- **Pilot B:** does one global conformal threshold under-cover noisy ECG, and does quality-stratified (Mondrian) calibration fix it without exploding set sizes? (PTB-XL)

**Run order:** keep `SMOKE = True` first (synthetic data, ~2 min) to confirm the code runs, then set `SMOKE = False`.

**Known simplifications (state them in any write-up):**
- Quantization here is *simulated weight-only* (round weights to a b-bit grid, keep float activations). It is not true integer inference.
- Pruning is global magnitude pruning with no fine-tuning.
- PathMNIST's test split comes from a different clinical center than train, so calibration and evaluation sets are both drawn from the test split (random halves) to keep them exchangeable.
- PTB-XL folds 1-8 train, 9 calibration, 10 test follow the dataset's recommended stratified folds (patient-level separation); this is from the dataset documentation as I remember it, so verify in its README.
- Noise is injected synthetically (white, baseline wander, muscle-like). Real artifact recordings (e.g. MIT-BIH NSTDB) are a later upgrade.

In [ ]:
import os, sys, copy, math, time, glob, subprocess, itertools
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F

SMOKE = True          # True = tiny synthetic data to test the code. Set False for the real run.
RUN_A = True          # set False to skip Pilot A (about 45 min) when you only need Pilot B
RUN_B = True
QUICK_B = False       # True = Pilot B with 1 seed, 1 noise draw, 8 epochs (fast first look)
ALPHA = 0.1           # target miscoverage (coverage 90%)
SEEDS = [0, 1, 2]     # training seeds
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PTBXL_ROOT = "/content/ptbxl"   # where PTB-XL is (or will be) downloaded.
# To survive Colab restarts, mount Drive first and use e.g. PTBXL_ROOT = "/content/drive/MyDrive/ptbxl"
# (run: from google.colab import drive; drive.mount("/content/drive"))
print("device:", DEVICE, "| smoke:", SMOKE)

def set_seed(s):
    np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

## Shared conformal code (split conformal, LAC and APS scores)

In [ ]:
def lac_all(probs):
    """Score of every label: 1 - p(label). probs: (n, K)."""
    return 1.0 - probs

def aps_all(probs):
    """Deterministic APS score of every label: total prob mass of labels ranked at or above it."""
    order = np.argsort(-probs, axis=1)
    cum = np.cumsum(np.take_along_axis(probs, order, axis=1), axis=1)
    ranks = np.argsort(order, axis=1)
    return np.take_along_axis(cum, ranks, axis=1)

SCORES = {"LAC": lac_all, "APS": aps_all}

def conformal_q(true_scores, alpha=ALPHA):
    """Finite-sample split-conformal quantile. Returns inf if the calibration set is too small."""
    n = len(true_scores)
    k = int(math.ceil((n + 1) * (1 - alpha)))
    if n == 0 or k > n:
        return np.inf
    return float(np.sort(true_scores)[k - 1])

def summarize(sets, y, min_class_n=10):
    covered = sets[np.arange(len(y)), y]
    cls = [covered[y == c].mean() for c in np.unique(y) if (y == c).sum() >= min_class_n]
    return dict(coverage=float(covered.mean()), size=float(sets.sum(1).mean()),
                worst_class_cov=float(min(cls)) if cls else np.nan)

## Shared model and compression code

In [ ]:
class Block(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.b1 = nn.BatchNorm2d(cout)
        self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False);     self.b2 = nn.BatchNorm2d(cout)
        self.sc = None
        if stride != 1 or cin != cout:
            self.sc = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
    def forward(self, x):
        o = F.relu(self.b1(self.c1(x))); o = self.b2(self.c2(o))
        return F.relu(o + (x if self.sc is None else self.sc(x)))

class ResNet18Small(nn.Module):
    """ResNet-18 layout (2,2,2,2 blocks) with a 3x3 stem, suited to 28x28 images."""
    def __init__(self, num_classes, width=64):
        super().__init__()
        w = width
        self.stem = nn.Sequential(nn.Conv2d(3, w, 3, 1, 1, bias=False), nn.BatchNorm2d(w), nn.ReLU())
        cfg, layers, cin = [(w, 1), (w, 1), (2*w, 2), (2*w, 1), (4*w, 2), (4*w, 1), (8*w, 2), (8*w, 1)], [], w
        for cout, s in cfg:
            layers.append(Block(cin, cout, s)); cin = cout
        self.layers = nn.Sequential(*layers); self.fc = nn.Linear(cin, num_classes)
    def forward(self, x):
        x = self.layers(self.stem(x))
        return self.fc(F.adaptive_avg_pool2d(x, 1).flatten(1))

class ECGNet(nn.Module):
    """Small 1D CNN for 12-lead ECG, input (N, 12, T)."""
    def __init__(self, num_classes, ch=(32, 64, 96, 128)):
        super().__init__()
        layers, cin = [], 12
        for c in ch:
            layers += [nn.Conv1d(cin, c, 7, 2, 3, bias=False), nn.BatchNorm1d(c), nn.ReLU()]; cin = c
        self.body = nn.Sequential(*layers); self.fc = nn.Linear(cin, num_classes)
    def forward(self, x):
        return self.fc(self.body(x).mean(-1))

def train_model(model, X, y, epochs, lr=2e-3, bs=256, seed=0):
    set_seed(seed); model.to(DEVICE).train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * math.ceil(len(y) / bs))
    Xt, yt = torch.as_tensor(X), torch.as_tensor(y).long()
    for ep in range(epochs):
        perm = torch.randperm(len(yt))
        for i in range(0, len(yt), bs):
            idx = perm[i:i + bs]
            xb, yb = Xt[idx].to(DEVICE), yt[idx].to(DEVICE)
            loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return model.eval()

@torch.no_grad()
def predict_probs(model, X, bs=512):
    model.to(DEVICE).eval(); out = []
    Xt = torch.as_tensor(X)
    for i in range(0, len(Xt), bs):
        out.append(F.softmax(model(Xt[i:i + bs].to(DEVICE)), dim=1).cpu())
    return torch.cat(out).numpy().astype(np.float64)

WEIGHT_LAYERS = (nn.Conv2d, nn.Conv1d, nn.Linear)

def quantize_weights(model, bits):
    """Simulated symmetric per-output-channel weight quantization (weights only)."""
    m, qmax = copy.deepcopy(model), 2 ** (bits - 1) - 1
    for mod in m.modules():
        if isinstance(mod, WEIGHT_LAYERS):
            w = mod.weight.data; dims = tuple(range(1, w.dim()))
            scale = w.abs().amax(dim=dims, keepdim=True).clamp_min(1e-8) / qmax
            mod.weight.data = torch.round(w / scale).clamp(-qmax, qmax) * scale
    return m

def prune_global(model, sparsity):
    import torch.nn.utils.prune as prune
    m = copy.deepcopy(model)
    params = [(mod, "weight") for mod in m.modules() if isinstance(mod, WEIGHT_LAYERS)]
    prune.global_unstructured(params, pruning_method=prune.L1Unstructured, amount=sparsity)
    for mod, name in params: prune.remove(mod, name)
    return m

COMPRESSIONS = {
    "fp32":        lambda m: m,
    "int8":        lambda m: quantize_weights(m, 8),
    "int4":        lambda m: quantize_weights(m, 4),
    "prune50":     lambda m: prune_global(m, 0.5),
    "prune70":     lambda m: prune_global(m, 0.7),
    "prune90":     lambda m: prune_global(m, 0.9),
    "int8+prune50": lambda m: prune_global(quantize_weights(m, 8), 0.5),
}

## Pilot A: reuse of the full-precision conformal threshold (PathMNIST)

In [ ]:
def load_pathmnist(smoke):
    if smoke:
        rng = np.random.RandomState(0); K = 9
        def make(n):
            y = rng.randint(0, K, n); X = rng.rand(n, 3, 28, 28).astype(np.float32) * 0.8
            for k in range(K):  # class-dependent blob position/colour so a tiny net can learn it
                r, c, ch = (k // 3) * 8 + 2, (k % 3) * 8 + 2, k % 3
                X[y == k, ch, r:r + 8, c:c + 8] += 0.6
            return (X - 0.5) / 0.5, y
        Xtr, ytr = make(3000); Xte, yte = make(1500)
        return Xtr, ytr, Xte, yte, K
    from medmnist import PathMNIST
    tr, te = PathMNIST(split="train", download=True), PathMNIST(split="test", download=True)
    f = lambda a: ((a.imgs.astype(np.float32) / 255.0 - 0.5) / 0.5).transpose(0, 3, 1, 2).copy()
    return f(tr), tr.labels.squeeze().astype(np.int64), f(te), te.labels.squeeze().astype(np.int64), 9

def pilot_A(smoke=SMOKE, seeds=SEEDS, n_splits=20, epochs=None, comps=None):
    Xtr, ytr, Xte, yte, K = load_pathmnist(smoke)
    epochs = epochs or (3 if smoke else 8)
    comps = comps or COMPRESSIONS
    rows = []
    for seed in seeds:
        t0 = time.time()
        model = train_model(ResNet18Small(K, width=16 if smoke else 64), Xtr, ytr, epochs, seed=seed)
        probs = {c: predict_probs(fn(model), Xte) for c, fn in comps.items()}
        print(f"seed {seed}: trained+compressed in {time.time()-t0:.0f}s | "
              + " ".join(f"{c}={np.mean(probs[c].argmax(1)==yte):.3f}" for c in comps))
        for sname, sfn in SCORES.items():
            S = {c: sfn(p) for c, p in probs.items()}
            for sp in range(n_splits):
                rng = np.random.RandomState(1000 * seed + sp)
                perm = rng.permutation(len(yte)); cal, ev = perm[:len(yte) // 2], perm[len(yte) // 2:]
                q_full = conformal_q(S["fp32"][cal, yte[cal]])
                for c in comps:
                    q_c = conformal_q(S[c][cal, yte[cal]])
                    for mode, q in (("reuse", q_full), ("recal", q_c)):
                        r = summarize(S[c][ev] <= q, yte[ev])
                        rows.append(dict(seed=seed, split=sp, score=sname, comp=c, mode=mode,
                                         acc=float(np.mean(probs[c].argmax(1) == yte)), **r))
    return pd.DataFrame(rows)

def report_A(df):
    if SMOKE: print("*** SMOKE = True: SYNTHETIC DATA, NOT A RESULT ***")
    g = (df.groupby(["score", "comp", "mode"])[["coverage", "size", "worst_class_cov", "acc"]].mean().reset_index())
    g["gap"] = g["coverage"] - (1 - ALPHA)
    piv = g.pivot_table(index=["score", "comp"], columns="mode", values=["coverage", "size", "worst_class_cov"])
    piv.columns = [f"{a}_{b}" for a, b in piv.columns]
    piv["gap_reuse_pts"] = 100 * (piv["coverage_reuse"] - (1 - ALPHA))
    piv["size_ratio_reuse_vs_recal"] = piv["size_reuse"] / piv["size_recal"]
    out = piv[["coverage_reuse", "coverage_recal", "gap_reuse_pts", "size_reuse", "size_recal",
               "size_ratio_reuse_vs_recal", "worst_class_cov_reuse", "worst_class_cov_recal"]].round(3)
    print(out.to_string())
    key = out.loc[(slice(None), ["int8", "prune50"]), "gap_reuse_pts"]
    worst = key.values[np.argmax(np.abs(key.values))]   # largest-magnitude gap (over- or under-coverage)
    print("\nPILOT A DECISION RULE (from the plan):")
    print(f"  largest INT8/prune50 reuse coverage gap = {worst:+.2f} points (negative = undercoverage, positive = overcoverage / larger sets)")
    print("  also compare size_ratio_reuse_vs_recal: reuse can keep coverage but inflate sets.")
    print("  -> CONTINUE topic 1" if abs(worst) >= 2 else
          "  -> gap below 2 points: weak effect; check int4/prune70 before giving up on topic 1"
          if abs(worst) >= 1 else "  -> gap under 1 point: switch away from topic 1")
    return out

In [ ]:
if RUN_A:
    dfA = pilot_A()
    resA = report_A(dfA)
    dfA.to_csv("pilotA_raw.csv", index=False)

## Pilot B: signal-quality-stratified conformal prediction on PTB-XL

In [ ]:
CLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]

def download_ptbxl(root=PTBXL_ROOT):
    """Downloads/resumes the 100 Hz records + metadata (~1 GB). Retries until every record in the CSV has its .hea file."""
    csv = os.path.join(root, "ptbxl_database.csv")
    n_hea = lambda: len(glob.glob(os.path.join(root, "records100", "*", "*_lr.hea")))
    def complete():
        return os.path.exists(csv) and n_hea() >= len(pd.read_csv(csv))
    if complete(): return
    os.makedirs(root, exist_ok=True)
    cmd = ("wget -nv -r -N -c -np -nH --cut-dirs=3 --tries=5 --waitretry=2 -X /files/ptb-xl/1.0.3/records500 "
           f"-P {root} https://physionet.org/files/ptb-xl/1.0.3/")
    for attempt in range(4):
        subprocess.run(cmd, shell=True)
        print(f"download attempt {attempt+1}: {n_hea()} .hea files in records100")
        if complete(): break
    print("records100 .hea files:", n_hea(), "| metadata rows:", len(pd.read_csv(csv)) if os.path.exists(csv) else "n/a")

def load_ptbxl(smoke, root=PTBXL_ROOT):
    if smoke:
        rng = np.random.RandomState(1); K = 5; T = 1000
        def make(n):
            y = rng.randint(0, K, n); t = np.arange(T) / 100.0
            X = np.zeros((n, 12, T), np.float32)
            for i in range(n):
                f = 1.0 + 0.35 * y[i]  # class-dependent "heart rate"
                base = np.sin(2 * np.pi * f * t) + 0.5 * np.sin(2 * np.pi * 2 * f * t)
                X[i] = base[None] * rng.uniform(0.7, 1.3, (12, 1)) + 0.15 * rng.randn(12, T)
            return X, y
        (Xtr, ytr), (Xca, yca), (Xte, yte) = make(2500), make(800), make(800)
        return Xtr, ytr, Xca, yca, Xte, yte, K
    import ast, wfdb
    download_ptbxl(root)
    df = pd.read_csv(os.path.join(root, "ptbxl_database.csv"), index_col="ecg_id")
    df["scp_codes"] = df["scp_codes"].apply(ast.literal_eval)
    agg = pd.read_csv(os.path.join(root, "scp_statements.csv"), index_col=0)
    agg = agg[agg["diagnostic"] == 1]
    sup = lambda codes: sorted({agg.loc[c, "diagnostic_class"] for c in codes if c in agg.index})
    df["sc"] = df["scp_codes"].apply(sup)
    df = df[df["sc"].apply(len) == 1].copy()                       # single-superclass records only
    df["label"] = df["sc"].apply(lambda s: CLASSES.index(s[0]))
    print("records kept:", len(df), "| class counts:", df["label"].value_counts().sort_index().tolist())
    for col in ["static_noise", "burst_noise", "baseline_drift", "electrodes_problems"]:
        if col in df.columns:
            print(f"metadata '{col}': {df[col].notna().sum()} annotated records")  # check if usable as real quality labels
    ok = df["filename_lr"].apply(lambda f: os.path.exists(os.path.join(root, f + ".hea")) and os.path.exists(os.path.join(root, f + ".dat")))
    print("records with missing files (dropped):", int((~ok).sum()))
    df = df[ok]
    assert len(df) > 10000, "too many missing files: re-run the cell to resume the download"
    X = np.stack([wfdb.rdsamp(os.path.join(root, f))[0].T for f in df["filename_lr"]]).astype(np.float32)
    fold, y = df["strat_fold"].values, df["label"].values
    mu, sd = X[fold <= 8].mean((0, 2), keepdims=True), X[fold <= 8].std((0, 2), keepdims=True) + 1e-6
    X = (X - mu) / sd
    sel = lambda m: (X[m], y[m])
    (Xtr, ytr), (Xca, yca), (Xte, yte) = sel(fold <= 8), sel(fold == 9), sel(fold == 10)
    return Xtr, ytr, Xca, yca, Xte, yte, len(CLASSES)

NOISE_LEVELS = [np.inf, 12.0, 6.0, 0.0]   # SNR in dB; inf = clean
LEVEL_NAMES = ["clean", "12dB", "6dB", "0dB"]

def add_noise(X, snr_db, rng, fs=100):
    """Add white / baseline-wander / muscle-like noise at a target SNR (per record)."""
    if not np.isfinite(snr_db): return X
    N, L, T = X.shape; t = np.arange(T) / fs
    kind = rng.randint(0, 3, N)
    n = np.zeros_like(X)
    for i in range(N):
        if kind[i] == 0:
            n[i] = rng.randn(L, T)
        elif kind[i] == 1:
            w = sum(np.sin(2 * np.pi * rng.uniform(0.05, 0.5) * t + rng.uniform(0, 6.28)) for _ in range(3))
            n[i] = w[None] * rng.uniform(0.5, 1.5, (L, 1))
        else:
            n[i] = np.diff(rng.randn(L, T + 1), axis=1)  # emphasises high frequencies
    n = n / (n.std(axis=(1, 2), keepdims=True) + 1e-9)
    sp = X.var(axis=(1, 2), keepdims=True)
    return (X + n * np.sqrt(sp / (10 ** (snr_db / 10)))).astype(np.float32)

def make_mixed(X, rng, probs=(0.4, 0.2, 0.2, 0.2)):
    lv = rng.choice(len(NOISE_LEVELS), size=len(X), p=probs)
    Xn = X.copy()
    for k, snr in enumerate(NOISE_LEVELS):
        m = lv == k
        if m.any(): Xn[m] = add_noise(X[m], snr, rng)
    return Xn, lv

def noise_index(X, fs=100):
    """Label-free quality proxy from the signal itself: high-frequency and low-frequency energy ratios."""
    v = X.var(axis=2) + 1e-9                                  # (N, L)
    hf = (np.diff(X, axis=2).var(axis=2) / (2 * v)).mean(1)
    k = fs
    lf = np.stack([np.apply_along_axis(lambda s: np.convolve(s, np.ones(k) / k, "same"), 1, X[i]).var(axis=1) / v[i]
                   for i in range(len(X))]).mean(1)
    return hf, lf

def groupwise_q(scores_true, groups, alpha=ALPHA):
    return {g: conformal_q(scores_true[groups == g], alpha) for g in np.unique(groups)}

def pilot_B(smoke=SMOKE, seeds=SEEDS, n_noise_draws=3, epochs=None):
    Xtr, ytr, Xca, yca, Xte, yte, K = load_ptbxl(smoke)
    epochs = epochs or (4 if smoke else 15)
    rows, count_rows = [], []
    for seed in seeds:
        t0 = time.time()
        model = train_model(ECGNet(K), Xtr, ytr, epochs, lr=3e-3, bs=128, seed=seed)
        print(f"seed {seed}: trained in {time.time()-t0:.0f}s (clean test acc {np.mean(predict_probs(model, Xte).argmax(1)==yte):.3f})")
        for d in range(n_noise_draws):
            rng = np.random.RandomState(7919 * seed + d)
            Xca_n, lca = make_mixed(Xca, rng); Xte_n, lte = make_mixed(Xte, rng)
            pca, pte = predict_probs(model, Xca_n), predict_probs(model, Xte_n)
            hf_ca, lf_ca = noise_index(Xca_n); hf_te, lf_te = noise_index(Xte_n)
            z = lambda a, ref: (a - ref.mean()) / (ref.std() + 1e-9)
            ni_ca = z(hf_ca, hf_ca) + z(lf_ca, lf_ca); ni_te = z(hf_te, hf_ca) + z(lf_te, lf_ca)
            edges = np.quantile(ni_ca, [0.25, 0.5, 0.75])
            bca, bte = np.digitize(ni_ca, edges), np.digitize(ni_te, edges)      # proxy quality bins (0=cleanest)
            clean_ca = lca == 0
            for sname, sfn in SCORES.items():
                Sca, Ste = sfn(pca), sfn(pte)
                tca = Sca[np.arange(len(yca)), yca]
                q_glob = conformal_q(tca)
                q_orc, q_prx = groupwise_q(tca, lca), groupwise_q(tca, bca)
                q_clean = conformal_q(tca[clean_ca])
                methods = {
                    "naive_mixed_cal":  np.full(len(yte), q_glob),
                    "naive_clean_cal":  np.full(len(yte), q_clean),
                    "mondrian_oracle":  np.array([q_orc[g] for g in lte]),
                    "mondrian_proxy":   np.array([q_prx[g] for g in bte]),
                }
                for mname, qv in methods.items():
                    sets = Ste <= qv[:, None]
                    for k, ln in enumerate(LEVEL_NAMES):
                        m = lte == k
                        r = summarize(sets[m], yte[m], min_class_n=5)
                        rows.append(dict(seed=seed, draw=d, score=sname, method=mname, level=ln, **r))
                if sname == "LAC" and d == 0:
                    count_rows.append(dict(seed=seed, **{f"cal_n_{ln}": int((lca == k).sum()) for k, ln in enumerate(LEVEL_NAMES)}))
    print("calibration counts per noise level (seed 0 draw 0):", count_rows[0] if count_rows else "n/a")
    return pd.DataFrame(rows)

def report_B(df):
    if SMOKE: print("*** SMOKE = True: SYNTHETIC DATA, NOT A RESULT ***")
    per = df.groupby(["score", "method", "level"])[["coverage", "size"]].mean().reset_index()
    per["level"] = pd.Categorical(per["level"], LEVEL_NAMES, ordered=True)
    cov = per.pivot_table(index=["score", "method"], columns="level", values="coverage", observed=True).round(3)
    siz = per.pivot_table(index=["score", "method"], columns="level", values="size", observed=True).round(2)
    print("COVERAGE by true noise level (target %.2f):" % (1 - ALPHA)); print(cov.to_string())
    print("\nMEAN SET SIZE by true noise level:"); print(siz.to_string())
    nv, mo = cov.loc[("APS", "naive_mixed_cal")], cov.loc[("APS", "mondrian_oracle")]
    mp = cov.loc[("APS", "mondrian_proxy")]
    under = (1 - ALPHA) - nv.min()
    print("\nPILOT B DECISION RULE (APS shown; check LAC too):")
    print(f"  naive worst-level undercoverage  = {100*under:.1f} pts (continue needs >= 5)")
    print(f"  mondrian_oracle worst-level gap   = {100*((1-ALPHA)-mo.min()):.1f} pts (want <= 2)")
    print(f"  mondrian_proxy  worst-level gap   = {100*((1-ALPHA)-mp.min()):.1f} pts (label-free quality bins)")
    print(f"  noisiest-level set size: naive {siz.loc[('APS','naive_mixed_cal'),'0dB']:.2f} | oracle-Mondrian {siz.loc[('APS','mondrian_oracle'),'0dB']:.2f} | max possible {len(CLASSES)}")
    ok = under >= 0.05 and ((1 - ALPHA) - mo.min()) <= 0.02 and siz.loc[("APS", "mondrian_oracle"), "0dB"] < 0.9 * len(CLASSES)
    print("  -> CONTINUE topic B" if ok else "  -> weak or unusable effect; see tables before deciding")
    return cov, siz

In [ ]:
if RUN_B:
    dfB = pilot_B(seeds=[0], n_noise_draws=1, epochs=8) if QUICK_B else pilot_B()
    covB, sizB = report_B(dfB)
    dfB.to_csv("pilotB_raw.csv", index=False)

## Gate (Nov 14)
Keep the topic whose pilot shows the clearer effect. Save `dfA` / `dfB` (`.to_csv`) so the protocol can be written from them.
If both pass: prefer B for novelty, A for lower technical risk. If neither passes: topic 2 was weakened by the EEG paper, so consider
a per-patient coverage metric on PTB-XL (see the doc) before committing.